# 02 · High-ticket machines

This notebook replicates the High-ticket machines paragraph of the report's Results section.

> High-ticket machines: We found 113 chronic outliers at 82 customers. The top 5% of the 6,621 telemetry devices hold 32.9% of their tickets vs. 22.2% by chance. Allowing for spread between machines, only 2 rate outliers still stand out, so high-ticket machines form a continuum. Non-success codes rise before their tickets no more than on other machines.

We also check the numbers behind two of its statements: the 118 rate outliers that "only 2" refers to (`docs/05_high_ticket_machines.md`), and the rise in non-success codes in each group, which the report's Definitions give as +20.3 for chronic outliers, +21.6 for other top-5% machines, and +21.5 for all other machines.

**How it works.** `high_ticket_machines.py` links IPA's case export to the clearView telemetry devices, gives each device an expected ticket count (its telemetry type's rate × its months active), flags rate outliers and chronic outliers, compares ticket concentration with 1,000 Poisson simulations, re-tests the rate outliers with a negative binomial, and summarizes non-success codes before each ticket. We recompute every number from the tables it writes, with one row per machine, device, or ticket. Only the chance benchmark comes from `htm_summary.json`, because it needs the script's simulations, and we redraw those simulations here as a cross-check.

**Re-run:** `verification/high_ticket_machines.py` (about 20 s). With `RERUN = True`, it rewrites its 13 `htm_*.csv` tables and `htm_summary.json` in a scratch copy of `verification/outputs/`, without its figures, and the notebook compares each file with the committed one.

**Used as committed, without re-running:** the script also reads four extracts of the telemetry database, `fu/fu_ticket_last10.csv` (each ticket's device and its 30-day windows), `fu/fu_devices.csv` (the device dictionary), `db/db_host_level.csv` (which host is IPA's shared internal host), and `db/db_machine_type_rates.csv` (the midterm analysis's rates by telemetry type, for comparison only). `db_followup.py` and `verify_with_db.py` wrote them from the telemetry database, which the notebooks do not re-read, so the re-run uses the committed files. This notebook also reads `fu/fu_devices.csv` and `db/db_host_level.csv` to rebuild the device population.

In [1]:
RERUN = True  # False: recompute from the committed outputs without re-running the scripts
import pandas as pd
import replicate as R
OUT = R.outputs(RERUN)

RERUN is True: the scripts write into a scratch copy of verification/outputs/.


In [2]:
if RERUN:
    R.rerun(OUT, 'high_ticket_machines.py')
R.compare(OUT)

high_ticket_machines.py: exit status 0, 18 s
    Done in 17.0 s; checks {'PASS': 33, 'FAIL': 0}
14 files written: 14 identical.


,file written by the re-run,compared with the committed file
0,htm_chronic_categories.csv,identical
1,htm_chronic_top3.csv,identical
2,htm_customer_chronic.csv,identical
3,htm_device_exposure.csv,identical
4,htm_episodes.csv,identical
5,htm_issue_mix.csv,identical
6,htm_lorenz.csv,identical
7,htm_machine_counts.csv,identical
8,htm_pre_ticket.csv,identical
9,htm_pre_ticket_summary.csv,identical


## The 6,621 telemetry devices

The exposure population is every device in the telemetry device table except the devices on IPA's shared internal host and the devices with no host. We rebuild it from `fu/fu_devices.csv` and `db/db_host_level.csv` and confirm that it is the set of devices in `htm_device_exposure.csv`, the script's table with one row per device.

In [3]:
devices = R.read_csv(OUT, 'fu/fu_devices.csv')  # one row per telemetry device (database extract)
hosts = R.read_csv(OUT, 'db/db_host_level.csv')
dev_exp = R.read_csv(OUT, 'htm_device_exposure.csv')  # one row per device in the exposure population

on_shared_host = devices.host.isin(hosts.host[hosts.shared_internal_host == 1])
no_host = devices.host.isna()
population = devices.machine_id[~on_shared_host & ~no_host]
n_devices = len(dev_exp)
print(f'htm_device_exposure.csv has {n_devices:,} devices, the same set as the population rebuilt here: '
      f'{set(dev_exp.machine_id) == set(population)}')
pd.Series({'telemetry devices': len(devices), 'on the shared internal host': int(on_shared_host.sum()),
           'with no host': int(no_host.sum()), 'exposure population': len(population)}, name='devices').to_frame()

htm_device_exposure.csv has 6,621 devices, the same set as the population rebuilt here: True


,devices
telemetry devices,7183
on the shared internal host,411
with no host,151
exposure population,6621


## 113 chronic outliers at 82 customers

A chronic outlier is a device in the top 5% of machines by unplanned tickets that is also a rate outlier. We take the top-5% cut from `htm_machine_counts.csv` (one row per ticketed machine: the count at rank 185, ties included) and recompute each device's Benjamini–Hochberg q-value from its Poisson p-value in `htm_device_exposure.csv` (a rate outlier has q < 0.05). The customer is the account named most often on the machine's tickets, as the report defines it for chronic outliers.

In [4]:
import math
import numpy as np


def bh(p):
    """Benjamini–Hochberg q-values, computed as in high_ticket_machines.py."""
    p = np.asarray(p, float)
    order = np.argsort(p, kind='mergesort')
    q = np.minimum.accumulate((p[order] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1]
    out = np.empty(len(p))
    out[order] = np.minimum(q, 1.0)
    return out


machines = R.read_csv(OUT, 'htm_machine_counts.csv')  # one row per machine with an unplanned ticket
rank_k = math.ceil(0.05 * len(machines))
cut = machines.unplanned_tickets.sort_values(ascending=False).iloc[rank_k - 1]
top5_machines = machines.machine_id[machines.unplanned_tickets >= cut]
top5 = dev_exp.serial_id.isin(top5_machines)

tested = dev_exp.expected > 0  # the script tests the devices with expected tickets above 0
q_poisson = pd.Series(np.nan, index=dev_exp.index)
q_poisson[tested] = bh(dev_exp.p_poisson[tested])
rate_outlier = q_poisson < 0.05

chronic = dev_exp[top5 & rate_outlier]
n_chronic = len(chronic)
n_chronic_customers = chronic.customer.nunique()
modal_account = chronic.machine_id.map(devices.set_index('machine_id').ticket_customer)
print(f'Top 5% by unplanned tickets: {cut} or more (the count at rank {rank_k} of {len(machines):,} machines), '
      f'{len(top5_machines)} machines with ties.')
print(f'{n_chronic} chronic outliers at {n_chronic_customers} customers; the customer is the account named most often '
      f'on the tickets for {(chronic.customer == modal_account).sum()} of the {n_chronic}.')
classes = pd.crosstab(top5.map({True: 'top 5% by unplanned tickets', False: 'other devices'}),
                      rate_outlier.map({True: 'rate outlier', False: 'not a rate outlier'}),
                      rownames=[None], colnames=[None], margins=True, margins_name='all')
classes.loc[['top 5% by unplanned tickets', 'other devices', 'all'], ['rate outlier', 'not a rate outlier', 'all']]

Top 5% by unplanned tickets: 5 or more (the count at rank 185 of 3,685 machines), 247 machines with ties.
113 chronic outliers at 82 customers; the customer is the account named most often on the tickets for 113 of the 113.


,rate outlier,not a rate outlier,all
top 5% by unplanned tickets,113,93,206
other devices,5,6410,6415
all,118,6503,6621


## The top 5% of the devices hold 32.9% of their tickets

We rank the 6,621 devices by their unplanned tickets from Aug 1st, 2025, to Jul 31st, 2026 (`tickets_obs` in `htm_device_exposure.csv`), take the top 5%, and divide their tickets by the devices' total.

In [5]:
tickets = dev_exp.tickets_obs.sort_values(ascending=False)
top_k = math.ceil(0.05 * n_devices)
top_share = tickets.iloc[:top_k].sum() / tickets.sum()
print(f'The top {top_k} of {n_devices:,} devices ({R.pct(top_k / n_devices, 1)}) hold {tickets.iloc[:top_k].sum():,} '
      f'of their {tickets.sum():,} tickets: {R.pct(top_share, 1)}.')

The top 332 of 6,621 devices (5.0%) hold 1,774 of their 5,395 tickets: 32.9%.


## 22.2% by chance

The script draws every device's tickets 1,000 times from a Poisson distribution with the device's expected tickets (its type's rate × its months active) as the mean and takes the median top-5% share over the draws. This needs the script's simulations, so we read the median from `htm_summary.json`. As a cross-check, we recompute each device's expected tickets from `htm_device_exposure.csv` and redraw the simulations with the script's seed, in the device order of `fu/fu_devices.csv` that the draws depend on.

In [6]:
summary = R.read_json(OUT, 'htm_summary.json')
d4 = summary['D4_concentration']
chance_median = d4['top5_share_null_median']

# Cross-check. A type's rate is its devices' tickets / months active (types under 100 months share one pooled rate,
# the rate_group column), and a device's expected tickets are its type's rate x its months active.
by_type = dev_exp.groupby('rate_group')[['tickets_obs', 'months']].sum()
expected = (dev_exp.rate_group.map(by_type.tickets_obs / by_type.months) * dev_exp.months).set_axis(dev_exp.machine_id)
draws = np.random.default_rng(summary['seed']).poisson(expected.reindex(population).to_numpy(),
                                                       size=(summary['n_sim'], len(population)))
top_draws = -np.sort(-draws, axis=1)[:, :top_k].sum(axis=1) / draws.sum(axis=1)
pd.DataFrame({'observed': [d4['top5_share_obs'], top_share],
              'chance: median': [chance_median, np.median(top_draws)],
              'chance: 95th percentile': [d4['top5_share_null_p95'], np.percentile(top_draws, 95)]},
             index=['htm_summary.json', 'recomputed here']).map(lambda x: R.pct(x, 2))

,observed,chance: median,chance: 95th percentile
htm_summary.json,32.88%,22.24%,22.81%
recomputed here,32.88%,22.24%,22.81%


## Only 2 of the 118 rate outliers still stand out under a negative binomial

The negative binomial allows each type's ticket rate to vary between machines. The script fits that spread for each type (`alpha_nb`), so we take each device's negative-binomial p-value from `htm_device_exposure.csv`, apply the same Benjamini–Hochberg adjustment over the same devices, and count the rate outliers that keep q < 0.05.

In [7]:
q_nb = pd.Series(np.nan, index=dev_exp.index)
q_nb[tested] = bh(dev_exp.p_nb[tested])
nb_kept = rate_outlier & (q_nb < 0.05)
n_rate_outliers, n_nb_kept = int(rate_outlier.sum()), int(nb_kept.sum())
rows = {'rate outliers (Poisson)': rate_outlier, 'still outliers under the negative binomial': nb_kept}
pd.DataFrame([{'devices': int(m.sum()), 'tickets': int(dev_exp.tickets_obs[m].sum()),
               'expected tickets': round(float(dev_exp.expected[m].sum()), 1),
               'median months active': float(dev_exp.months[m].median())} for m in rows.values()],
             index=list(rows))

,devices,tickets,expected tickets,median months active
rate outliers (Poisson),118,837,98.9,12.0
still outliers under the negative binomial,2,24,0.7,4.0


## Non-success codes rise no more before chronic outliers' tickets

`htm_pre_ticket.csv` has one row per usable unplanned ticket, with its transactions and non-success codes in the 30 days before the ticket and in its comparison window, the same machine's 30 days ending 35 days before the ticket. For the tickets with a clean comparison window, we compute non-success codes per 1,000 transactions in each window, and the rise is the median of the ticket windows minus the median of the comparison windows in each group (chronic outliers, other top-5% machines, and all other machines). Windows with no transactions have no rate and are left out of the medians.

In [8]:
pre = R.read_csv(OUT, 'htm_pre_ticket.csv')  # one row per usable unplanned ticket
paired = pre[pre.control_clean_available]


def per_1000(non_success, events):
    return (1000 * non_success / events).where(events > 0)


groups = {'chronic': 'chronic outliers', 'workload_outsized': 'other top-5% machines', 'other': 'all other machines'}
by_group = (paired.assign(ticket=per_1000(paired.non_success_events_30d, paired.all_events_30d),
                          comparison=per_1000(paired.control_non_success_events_30d, paired.control_all_events_30d))
            .groupby('group').agg(**{'paired tickets': ('case_id', 'size'), 'machines': ('machine_id', 'nunique'),
                                     'median, ticket window': ('ticket', 'median'),
                                     'median, comparison window': ('comparison', 'median')})
            .loc[list(groups)].rename(index=groups).rename_axis(None))
by_group['rise'] = by_group['median, ticket window'] - by_group['median, comparison window']
rise_chronic, rise_top5, rise_other = by_group['rise']
print(f"The chronic outliers' rise ({rise_chronic:+.1f}) is no larger than the other groups' "
      f"({rise_top5:+.1f} and {rise_other:+.1f}): {rise_chronic <= min(rise_top5, rise_other)}")
by_group.round(1)

The chronic outliers' rise (+20.3) is no larger than the other groups' (+21.6 and +21.5): True


,paired tickets,machines,"median, ticket window","median, comparison window",rise
chronic outliers,163,98,68.8,48.5,20.3
other top-5% machines,169,84,76.5,54.9,21.6
all other machines,2927,2322,62.9,41.4,21.5


## Check against the report

In [9]:
R.check([
    ('Chronic outliers', '113', R.num(n_chronic)),
    ('Customers with a chronic outlier', '82', R.num(n_chronic_customers)),
    ('Top group as a share of the devices', '5%', R.pct(top_k / n_devices)),
    ('Telemetry devices in the exposure population', '6,621', R.num(n_devices)),
    ("Top 5%'s share of the devices' tickets", '32.9%', R.pct(top_share, 1)),
    ('Same share by chance (median of 1,000 simulations)', '22.2%', R.pct(chance_median, 1)),
    ('Rate outliers before allowing for spread (docs/05)', '118', R.num(n_rate_outliers)),
    ('Rate outliers that still stand out under the negative binomial', '2', R.num(n_nb_kept)),
    ('Rise in non-success codes per 1,000: chronic outliers (Definitions)', '+20.3', f'{rise_chronic:+.1f}'),
    ('Rise: other top-5% machines (Definitions)', '+21.6', f'{rise_top5:+.1f}'),
    ('Rise: all other machines (Definitions)', '+21.5', f'{rise_other:+.1f}'),
    ("Chronic outliers' rise vs. other machines'", 'no more than',
     'no more than' if rise_chronic <= min(rise_top5, rise_other) else 'more than'),
]);

,Finding,Report,Recomputed,Match
0,Chronic outliers,113,113,yes
1,Customers with a chronic outlier,82,82,yes
2,Top group as a share of the devices,5%,5%,yes
3,Telemetry devices in the exposure population,"6,621","6,621",yes
4,Top 5%'s share of the devices' tickets,32.9%,32.9%,yes
5,"Same share by chance (median of 1,000 simulati...",22.2%,22.2%,yes
6,Rate outliers before allowing for spread (docs...,118,118,yes
7,Rate outliers that still stand out under the n...,2,2,yes
8,"Rise in non-success codes per 1,000: chronic o...",+20.3,+20.3,yes
9,Rise: other top-5% machines (Definitions),+21.6,+21.6,yes


All 12 values match the report.
